In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

# Files uploaded to Colab
protein_file = "ProCan-DepMapSanger_protein_matrix_8498_averaged.txt"
mapping_file = "ProCan-DepMapSanger_mapping_file_averaged.txt"
expression_file = "gene_attribute_matrix (1).txt.gz"
mutation_file = "gene_attribute_matrix (2).txt.gz"
cnv_file = "gene_attribute_matrix (3).txt.gz"
overlap_file = "All_Omics_GDSC_Provisional_Overlap.csv"

# Check that every file exists
for filename in [
    protein_file, mapping_file, expression_file,
    mutation_file, cnv_file, overlap_file
]:
    print(filename, "FOUND" if Path(filename).exists() else "MISSING")

# Load the smaller metadata files
mapping = pd.read_csv(mapping_file, sep="\t")
overlap = pd.read_csv(overlap_file)

print("\nMapping shape:", mapping.shape)
print("Provisional overlap shape:", overlap.shape)


ProCan-DepMapSanger_protein_matrix_8498_averaged.txt FOUND
ProCan-DepMapSanger_mapping_file_averaged.txt FOUND
gene_attribute_matrix (1).txt.gz FOUND
gene_attribute_matrix (2).txt.gz FOUND
gene_attribute_matrix (3).txt.gz FOUND
All_Omics_GDSC_Provisional_Overlap.csv FOUND

Mapping shape: (949, 6)
Provisional overlap shape: (526, 7)


In [3]:

import pandas as pd
import gzip
import csv

# Define the filenames
expression_file = "gene_attribute_matrix (1).txt.gz"
mutation_file = "gene_attribute_matrix (2).txt.gz"
cnv_file = "gene_attribute_matrix (3).txt.gz"

# Function to inspect each omics dataset
def inspect_omics(filename, dataset_name):
    with gzip.open(filename, "rt") as f:
        reader = csv.reader(f, delimiter="\t")

        header = next(reader)
        second_row = next(reader)
        third_row = next(reader)
        first_data_row = next(reader)

    print(f"\n--- {dataset_name} ---")
    print("Total columns:", len(header))
    print("Cell-line columns:", len(header) - 3)
    print("First 5 cell lines:", header[3:8])
    print("First gene:", first_data_row[0])
    print("First 5 values:", first_data_row[3:8])

# Inspect all three datasets
inspect_omics(expression_file, "Gene Expression")
inspect_omics(mutation_file, "Mutation")
inspect_omics(cnv_file, "Copy Number")


--- Gene Expression ---
Total columns: 1040
Cell-line columns: 1037
First 5 cell lines: ['CHL1', 'HMCB', 'HS852T', 'HS695T', 'A101D']
First gene: FBN1
First 5 values: ['0.000000', '0.000000', '0.000000', '-0.000000', '0.000000']

--- Mutation ---
Total columns: 907
Cell-line columns: 904
First 5 cell lines: ['HEL', 'HEL9217', 'TCCPAN2', 'SW1463', 'GOS3']
First gene: NCAM1
First 5 values: ['1.000000', '1.000000', '1.000000', '1.000000', '1.000000']

--- Copy Number ---
Total columns: 1046
Cell-line columns: 1043
First 5 cell lines: ['P3HR1', 'RCC10RGB', 'T84', 'KMS34', 'A4FUK']
First gene: SNORD116-3
First 5 values: ['-0.000000', '0.000000', '-0.000000', '0.000000', '0.000000']


In [4]:

import pandas as pd
import numpy as np
import re

# Load provisional overlap list
overlap = pd.read_csv(
    "All_Omics_GDSC_Provisional_Overlap.csv"
)

# Normalize names for provisional matching
def simplify_name(name):
    return re.sub(r"[\s\-_]", "", str(name).upper())

# Load each Harmonizome matrix
# First three rows contain header/metadata information
def load_omics(filename):
    df = pd.read_csv(
        filename,
        sep="\t",
        compression="gzip",
        header=None,
        skiprows=3
    )

    return df

expression = load_omics("gene_attribute_matrix (1).txt.gz")
mutation = load_omics("gene_attribute_matrix (2).txt.gz")
cnv = load_omics("gene_attribute_matrix (3).txt.gz")

print("Expression shape:", expression.shape)
print("Mutation shape:", mutation.shape)
print("CNV shape:", cnv.shape)
print("Provisional overlap:", overlap.shape)


Expression shape: (18025, 1040)
Mutation shape: (1667, 907)
CNV shape: (23264, 1046)
Provisional overlap: (526, 7)


In [5]:

import gzip
import csv
import pandas as pd
from collections import Counter

# Get the original cell-line names from each file
def get_cell_names(filename):
    with gzip.open(filename, "rt") as f:
        reader = csv.reader(f, delimiter="\t")
        header = next(reader)

    return header[3:]

expression_names = get_cell_names(expression_file)
mutation_names = get_cell_names(mutation_file)
cnv_names = get_cell_names(cnv_file)

# Use the original names in the provisional overlap file
target = overlap[["SIDM", "Cell_line"]].copy()
target["simple_name"] = target["Cell_line"].map(simplify_name)

# Find unique formatting-compatible matches
def align_names(names, target):
    counts = Counter(simplify_name(name) for name in names)

    lookup = {}
    for i, name in enumerate(names):
        key = simplify_name(name)
        if counts[key] == 1:
            lookup[key] = i

    aligned = target.copy()
    aligned["column_position"] = aligned["simple_name"].map(lookup)

    return aligned

expression_alignment = align_names(expression_names, target)
mutation_alignment = align_names(mutation_names, target)
cnv_alignment = align_names(cnv_names, target)

print("Expression matched:",
      expression_alignment["column_position"].notna().sum())

print("Mutation matched:",
      mutation_alignment["column_position"].notna().sum())

print("CNV matched:",
      cnv_alignment["column_position"].notna().sum())

# Identify samples matched in all three datasets
shared_ids = (
    set(expression_alignment.dropna(subset=["column_position"])["SIDM"])
    & set(mutation_alignment.dropna(subset=["column_position"])["SIDM"])
    & set(cnv_alignment.dropna(subset=["column_position"])["SIDM"])
)

print("Shared cell lines:", len(shared_ids))


Expression matched: 526
Mutation matched: 526
CNV matched: 526
Shared cell lines: 526


In [6]:

import pandas as pd
import numpy as np

# Keep the same sample order across all datasets
sample_order = target[
    target["SIDM"].isin(shared_ids)
]["SIDM"].tolist()

# Extract and transpose each matrix
def create_aligned_matrix(df, alignment, sample_order):

    # Match the sample order
    aligned = (
        alignment.set_index("SIDM")
        .loc[sample_order]
    )

    # Column positions correspond to cell-line columns
    positions = aligned["column_position"].astype(int).tolist()

    # First three columns contain gene metadata
    selected = df.iloc[:, [0] + [p + 3 for p in positions]].copy()

    # First column contains gene identifiers
    gene_names = selected.iloc[:, 0].astype(str)

    values = selected.iloc[:, 1:].apply(
        pd.to_numeric, errors="coerce"
    )

    # Transpose to cell lines x genes
    matrix = values.T
    matrix.columns = gene_names
    matrix.index = sample_order
    matrix.index.name = "SIDM"

    return matrix

expression_aligned = create_aligned_matrix(
    expression, expression_alignment, sample_order
)

mutation_aligned = create_aligned_matrix(
    mutation, mutation_alignment, sample_order
)

cnv_aligned = create_aligned_matrix(
    cnv, cnv_alignment, sample_order
)

print("Expression:", expression_aligned.shape)
print("Mutation:", mutation_aligned.shape)
print("CNV:", cnv_aligned.shape)

print(
    "All sample orders match:",
    expression_aligned.index.equals(mutation_aligned.index)
    and mutation_aligned.index.equals(cnv_aligned.index)
)


Expression: (526, 18025)
Mutation: (526, 1667)
CNV: (526, 23264)
All sample orders match: True


In [7]:

import pandas as pd
import numpy as np

datasets = {
    "Gene Expression": expression_aligned,
    "Mutation": mutation_aligned,
    "Copy Number": cnv_aligned
}

for name, df in datasets.items():
    print(f"\n--- {name} ---")

    # Check dimensions
    print("Shape:", df.shape)

    # Check duplicate gene names
    print("Duplicate gene columns:",
          df.columns.duplicated().sum())

    # Check missing values
    missing = df.isna().sum().sum()
    print("Missing values:", missing)
    print("Missing percentage:",
          round(missing / df.size * 100, 2))

    # Check features that have the same value in every cell line
    constant_features = (
        df.nunique(dropna=True) <= 1
    ).sum()

    print("Constant features:", constant_features)

    # Check the values represented in the matrix
    values = pd.Series(df.to_numpy().ravel()).value_counts()

    print("Value frequencies:")
    print(values.head(10))



--- Gene Expression ---
Shape: (526, 18025)
Duplicate gene columns: 0
Missing values: 0
Missing percentage: 0.0
Constant features: 13
Value frequencies:
-0.0    9118678
 1.0     209344
-1.0     153128
Name: count, dtype: int64

--- Mutation ---
Shape: (526, 1667)
Duplicate gene columns: 0
Missing values: 0
Missing percentage: 0.0
Constant features: 5
Value frequencies:
0.0    814194
1.0     62648
Name: count, dtype: int64

--- Copy Number ---
Shape: (526, 23264)
Duplicate gene columns: 0
Missing values: 0
Missing percentage: 0.0
Constant features: 0
Value frequencies:
-0.0    11597987
 1.0      323461
-1.0      315416
Name: count, dtype: int64


In [8]:

import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler

class OmicsPreprocessor:
    def __init__(self, scale=False):
        self.scale = scale
        self.selector = VarianceThreshold(threshold=0.0)
        self.scaler = StandardScaler() if scale else None

    def fit(self, X_train):
        # Learn which features vary in training data
        X_selected = self.selector.fit_transform(X_train)

        # Learn scaling parameters from training data only
        if self.scale:
            self.scaler.fit(X_selected)

        return self

    def transform(self, X):
        X_selected = self.selector.transform(X)

        if self.scale:
            return self.scaler.transform(X_selected)

        return X_selected

    def fit_transform(self, X_train):
        self.fit(X_train)
        return self.transform(X_train)

# Define preprocessors for each omics layer
expression_preprocessor = OmicsPreprocessor(scale=True)
mutation_preprocessor = OmicsPreprocessor(scale=False)
cnv_preprocessor = OmicsPreprocessor(scale=False)

print("Preprocessing pipelines initialized.")
print("Expression: variance filtering + scaling")
print("Mutation: variance filtering, binary values preserved")
print("CNV: variance filtering, discrete values preserved")


Preprocessing pipelines initialized.
Expression: variance filtering + scaling
Mutation: variance filtering, binary values preserved
CNV: variance filtering, discrete values preserved


In [9]:

import pandas as pd
import numpy as np

# Load the original proteomics matrix
protein_data = pd.read_csv(
    "ProCan-DepMapSanger_protein_matrix_8498_averaged.txt",
    sep="\t"
)

# Load the cell-line mapping file
protein_mapping = pd.read_csv(
    "ProCan-DepMapSanger_mapping_file_averaged.txt",
    sep="\t"
)

# Load the provisional overlap list
overlap = pd.read_csv(
    "All_Omics_GDSC_Provisional_Overlap.csv"
)

# Keep only the 526 provisionally overlapping cell lines
protein_subset = protein_data[
    protein_data["Project_Identifier"].isin(
        overlap["Project_Identifier"]
    )
].copy()

# Extract protein abundance measurements
protein_values = protein_subset.set_index("Project_Identifier")

# Match the ordering of the overlap file
protein_values = protein_values.loc[
    overlap["Project_Identifier"]
]

# Replace combined identifiers with SIDM IDs
protein_values.index = overlap["SIDM"].values
protein_values.index.name = "SIDM"

# Verify the subset
print("Proteomics subset shape:", protein_values.shape)
print("Unique cell lines:", protein_values.index.nunique())

print(
    "Missing percentage:",
    round(protein_values.isna().sum().sum() /
          protein_values.size * 100, 2)
)

print(
    "Sample order matches expression:",
    protein_values.index.equals(expression_aligned.index)
)

Proteomics subset shape: (526, 8498)
Unique cell lines: 526
Missing percentage: 38.84
Sample order matches expression: True


In [10]:

import pandas as pd
import numpy as np

# Calculate the proportion of cell lines with measurements
# for each protein
protein_coverage = protein_values.notna().mean(axis=0)

print("Total proteins:", len(protein_coverage))

print("\nProtein coverage summary:")
print(protein_coverage.describe())

# Compare different coverage thresholds
for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
    count = (protein_coverage >= threshold).sum()

    print(
        f"Proteins with at least {threshold*100:.0f}% coverage:",
        count
    )

# Identify proteins with no available measurements
print(
    "\nProteins with 0% coverage:",
    (protein_coverage == 0).sum()
)


Total proteins: 8498

Protein coverage summary:
count    8498.000000
mean        0.611573
std         0.351035
min         0.000000
25%         0.275665
50%         0.680608
75%         0.969582
max         1.000000
dtype: float64
Proteins with at least 50% coverage: 5172
Proteins with at least 60% coverage: 4681
Proteins with at least 70% coverage: 4156
Proteins with at least 80% coverage: 3648
Proteins with at least 90% coverage: 2960

Proteins with 0% coverage: 14


In [11]:

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

class ProteomicsPreprocessor:
    def __init__(self, min_coverage=0.70):
        self.min_coverage = min_coverage
        self.imputer = SimpleImputer(strategy="median")
        self.scaler = StandardScaler()

    def fit(self, X_train):
        # Use training data to identify well-measured proteins
        coverage = X_train.notna().mean(axis=0)

        self.selected_proteins = coverage[
            coverage >= self.min_coverage
        ].index.tolist()

        if len(self.selected_proteins) == 0:
            raise ValueError("No proteins meet the coverage threshold.")

        # Learn imputation values using training samples
        X_selected = X_train[self.selected_proteins]

        X_imputed = self.imputer.fit_transform(X_selected)

        # Learn scaling parameters
        self.scaler.fit(X_imputed)

        return self

    def transform(self, X):
        # Apply the same selected proteins and learned statistics
        X_selected = X[self.selected_proteins]

        X_imputed = self.imputer.transform(X_selected)

        X_scaled = self.scaler.transform(X_imputed)

        return pd.DataFrame(
            X_scaled,
            index=X.index,
            columns=self.selected_proteins
        )

    def fit_transform(self, X_train):
        self.fit(X_train)
        return self.transform(X_train)

# Initialize without fitting on the whole dataset
proteomics_preprocessor = ProteomicsPreprocessor(
    min_coverage=0.70
)

print("Proteomics preprocessing pipeline initialized.")
print("Minimum protein coverage: 70%")
print("Missing-value handling: median imputation")
print("Scaling: standardization")


Proteomics preprocessing pipeline initialized.
Minimum protein coverage: 70%
Missing-value handling: median imputation
Scaling: standardization


In [12]:

from pathlib import Path
import pandas as pd

# Create a folder for intermediate outputs
output_dir = Path("aligned_omics")
output_dir.mkdir(exist_ok=True)

# Export matrices without fitting preprocessing
matrices = {
    "expression": expression_aligned,
    "mutation": mutation_aligned,
    "cnv": cnv_aligned,
    "proteomics": protein_values
}

for name, df in matrices.items():
    df.to_csv(output_dir / f"{name}_aligned.csv")
    print(f"Saved {name}: {df.shape}")

# Export sample metadata
sample_metadata = overlap.set_index("SIDM").loc[
    expression_aligned.index
]

sample_metadata.to_csv(
    output_dir / "sample_metadata.csv"
)

# Verify all sample orders
reference_index = expression_aligned.index

print("\nAll sample orders identical:",
      all(df.index.equals(reference_index)
          for df in matrices.values()))

print("Output folder:", output_dir)


Saved expression: (526, 18025)
Saved mutation: (526, 1667)
Saved cnv: (526, 23264)
Saved proteomics: (526, 8498)

All sample orders identical: True
Output folder: aligned_omics


In [13]:

# Candidate sex-associated marker genes from the research proposal
xy_gene_candidates = [
    "XIST",
    "RPS4Y1",
    "DDX3Y",
    "KDM5D",
    "UTY"
]

# Check which markers exist in the expression matrix
available_xy_genes = [
    gene for gene in xy_gene_candidates
    if gene in expression_aligned.columns
]

missing_xy_genes = [
    gene for gene in xy_gene_candidates
    if gene not in expression_aligned.columns
]

print("Candidate X/Y genes:", len(xy_gene_candidates))
print("Available in expression:", available_xy_genes)
print("Missing from expression:", missing_xy_genes)

# Export the available markers
pd.DataFrame({
    "Gene": available_xy_genes
}).to_csv(
    "aligned_omics/XY_gene_module_candidates.csv",
    index=False
)


Candidate X/Y genes: 5
Available in expression: ['XIST', 'RPS4Y1', 'DDX3Y', 'KDM5D', 'UTY']
Missing from expression: []


In [14]:

from pathlib import Path

# Ensure the output folder exists
output_dir = Path("aligned_omics")
output_dir.mkdir(parents=True, exist_ok=True)

# Save the matched sex labels from the previous analysis
matched.to_csv(
    output_dir / "sex_labels_matched.csv",
    index=False
)

print("Saved successfully!")
print("Filename: sex_labels_matched.csv")
print("Rows:", len(matched))
print("Usable sex labels:", matched["usable_sex"].sum())


Saved successfully!
Filename: sex_labels_matched.csv
Rows: 526
Usable sex labels: 492


In [15]:

import pandas as pd

# Load matched sex metadata
sex_data = pd.read_csv(
    "aligned_omics/sex_labels_matched.csv"
)

# Keep records eligible for sex analysis
usable = sex_data[
    sex_data["usable_sex"] == True
].copy()

# Ensure each cell line appears only once
usable = usable.drop_duplicates(subset="SIDM")

# Match the labels to both molecular matrices
shared_ids = (
    set(usable["SIDM"])
    & set(expression_aligned.index)
    & set(protein_values.index)
)

# Preserve the original sample order
sample_ids = [
    sidm for sidm in expression_aligned.index
    if sidm in shared_ids
]

# Extract molecular data
expression_sex = expression_aligned.loc[sample_ids]
proteomics_sex = protein_values.loc[sample_ids]

# Align labels to the same sample order
sex_metadata = (
    usable.set_index("SIDM")
    .loc[sample_ids]
)

print("Matched cell lines:", len(sample_ids))

print("\nSex distribution:")
print(sex_metadata["final_sex"].value_counts())

print("\nExpression shape:", expression_sex.shape)
print("Proteomics shape:", proteomics_sex.shape)

print(
    "All sample orders match:",
    expression_sex.index.equals(proteomics_sex.index)
    and expression_sex.index.equals(sex_metadata.index)
)


Matched cell lines: 492

Sex distribution:
final_sex
Male      267
Female    225
Name: count, dtype: int64

Expression shape: (492, 18025)
Proteomics shape: (492, 8498)
All sample orders match: True


In [16]:

import pandas as pd

# Count male and female cell lines for each tissue
tissue_sex_counts = pd.crosstab(
    sex_metadata["Tissue_type"].fillna("Unknown"),
    sex_metadata["final_sex"]
)

# Ensure both columns exist
tissue_sex_counts = tissue_sex_counts.reindex(
    columns=["Male", "Female"], fill_value=0
)

tissue_sex_counts["Total"] = (
    tissue_sex_counts["Male"] +
    tissue_sex_counts["Female"]
)

# Identify tissues with at least 5 lines per sex
tissue_sex_counts["Eligible_for_comparison"] = (
    (tissue_sex_counts["Male"] >= 5) &
    (tissue_sex_counts["Female"] >= 5)
)

print("Tissue categories:", len(tissue_sex_counts))
print("Tissues with >=5 samples per sex:",
      tissue_sex_counts["Eligible_for_comparison"].sum())

display(
    tissue_sex_counts.sort_values(
        "Total", ascending=False
    )
)

# Save for research documentation
tissue_sex_counts.to_csv(
    "aligned_omics/tissue_sex_counts.csv"
)


Tissue categories: 21
Tissues with >=5 samples per sex: 9


final_sex,Male,Female,Total,Eligible_for_comparison
Tissue_type,,,,
Lung,83,34,117,True
Haematopoietic and Lymphoid,45,33,78,True
Breast,0,39,39,False
Large Intestine,19,13,32,True
Central Nervous System,15,11,26,True
Skin,13,12,25,True
Ovary,0,24,24,False
Esophagus,14,6,20,True
Pancreas,13,5,18,True


In [17]:

from pathlib import Path
import pandas as pd

# Identify tissues with at least 5 samples per sex
eligible_tissues = tissue_sex_counts.index[
    tissue_sex_counts["Eligible_for_comparison"]
].tolist()

# Select cell lines from eligible tissues
analysis_metadata = sex_metadata[
    sex_metadata["Tissue_type"].isin(eligible_tissues)
].copy()

# Align expression and proteomics
analysis_ids = analysis_metadata.index.tolist()

expression_analysis = expression_sex.loc[analysis_ids]
proteomics_analysis = proteomics_sex.loc[analysis_ids]

# Print results
print("Eligible tissues:", len(eligible_tissues))
print("Total analysis samples:", len(analysis_ids))

print("\nSex distribution:")
print(analysis_metadata["final_sex"].value_counts())

print("\nLabel confidence:")
print(analysis_metadata["label_confidence"].value_counts())

# Check high-confidence samples
high_confidence = analysis_metadata[
    analysis_metadata["label_confidence"] == "high"
]

print("\nHigh-confidence samples by tissue and sex:")
display(
    pd.crosstab(
        high_confidence["Tissue_type"],
        high_confidence["final_sex"]
    )
)

print("\nExpression shape:", expression_analysis.shape)
print("Proteomics shape:", proteomics_analysis.shape)

# Save the analysis subset metadata
output_dir = Path("aligned_omics")
output_dir.mkdir(parents=True, exist_ok=True)

analysis_metadata.to_csv(
    output_dir / "sex_differential_analysis_samples.csv"
)

print("\nAnalysis metadata saved successfully!")


Eligible tissues: 9
Total analysis samples: 344

Sex distribution:
final_sex
Male      218
Female    126
Name: count, dtype: int64

Label confidence:
label_confidence
low       172
high      171
medium      1
Name: count, dtype: int64

High-confidence samples by tissue and sex:


final_sex,Female,Male
Tissue_type,,
Bladder,1,4
Central Nervous System,7,11
Esophagus,0,2
Haematopoietic and Lymphoid,17,32
Large Intestine,5,10
Lung,16,41
Pancreas,1,2
Skin,7,9
Stomach,4,2



Expression shape: (344, 18025)
Proteomics shape: (344, 8498)

Analysis metadata saved successfully!


In [18]:

import pandas as pd

# Keep only high-confidence labels
high_conf_metadata = analysis_metadata[
    analysis_metadata["label_confidence"] == "high"
].copy()

# Count male/female samples per tissue
high_conf_counts = pd.crosstab(
    high_conf_metadata["Tissue_type"],
    high_conf_metadata["final_sex"]
).reindex(columns=["Male", "Female"], fill_value=0)

# Require at least 5 samples per sex
high_conf_tissues = high_conf_counts.index[
    (high_conf_counts["Male"] >= 5) &
    (high_conf_counts["Female"] >= 5)
]

high_conf_metadata = high_conf_metadata[
    high_conf_metadata["Tissue_type"].isin(high_conf_tissues)
].copy()

# Align molecular data
high_conf_ids = high_conf_metadata.index.tolist()

expression_high_conf = expression_analysis.loc[high_conf_ids]
proteomics_high_conf = proteomics_analysis.loc[high_conf_ids]

print("High-confidence eligible tissues:",
      len(high_conf_tissues))

print("High-confidence analysis samples:",
      len(high_conf_metadata))

print("\nSex distribution:")
print(high_conf_metadata["final_sex"].value_counts())

print("\nEligible tissue counts:")
display(high_conf_counts.loc[high_conf_tissues])

print("\nExpression:", expression_high_conf.shape)
print("Proteomics:", proteomics_high_conf.shape)


High-confidence eligible tissues: 5
High-confidence analysis samples: 155

Sex distribution:
final_sex
Male      103
Female     52
Name: count, dtype: int64

Eligible tissue counts:


final_sex,Male,Female
Tissue_type,,
Central Nervous System,11,7
Haematopoietic and Lymphoid,32,17
Large Intestine,10,5
Lung,41,16
Skin,9,7



Expression: (155, 18025)
Proteomics: (155, 8498)


In [19]:

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

# Use the high-confidence subset from Step 16
X = proteomics_high_conf.copy()
meta = high_conf_metadata.loc[X.index].copy()

# Female = 1, Male = 0
female = (meta["final_sex"] == "Female").astype(float)

# Add tissue indicators to control for tissue type
tissue = pd.get_dummies(
    meta["Tissue_type"],
    drop_first=True,
    dtype=float
)

design = pd.concat(
    [female.rename("Female"), tissue],
    axis=1
)

design = sm.add_constant(design)

results = []

# Test each protein separately
for protein in X.columns:
    y = pd.to_numeric(X[protein], errors="coerce")

    valid = y.notna()

    # Require enough measurements in both sexes
    if valid.sum() < 20:
        continue

    if ((valid & (female == 1)).sum() < 10 or
        (valid & (female == 0)).sum() < 10):
        continue

    d = design.loc[valid]
    response = y.loc[valid]

    # Skip constant proteins or non-estimable models
    if response.nunique() < 2:
        continue

    if np.linalg.matrix_rank(d.to_numpy()) < d.shape[1]:
        continue

    model = sm.OLS(response, d).fit()

    results.append({
        "Protein": protein,
        "n": int(valid.sum()),
        "n_male": int((valid & (female == 0)).sum()),
        "n_female": int((valid & (female == 1)).sum()),
        "Female_minus_Male": model.params["Female"],
        "p_value": model.pvalues["Female"]
    })

results_df = pd.DataFrame(results)

if results_df.empty:
    raise ValueError("No proteins passed the analysis filters.")

# Correct for testing many proteins
results_df["FDR"] = multipletests(
    results_df["p_value"],
    method="fdr_bh"
)[1]

results_df = results_df.sort_values("FDR")

print("Proteins tested:", len(results_df))
print("Proteins with FDR < 0.05:",
      (results_df["FDR"] < 0.05).sum())

display(results_df.head(20))

# Export results
results_df.to_csv(
    "aligned_omics/sex_differential_proteins.csv",
    index=False
)


Proteins tested: 6705
Proteins with FDR < 0.05: 0


,Protein,n,n_male,n_female,Female_minus_Male,p_value,FDR
4133,O00571;DDX3X_HUMAN,154,102,52,0.426344,0.000089,0.599240
4461,Q15651;HMGN3_HUMAN,47,32,15,0.037523,0.896612,0.998928
4474,Q86X83;COMD2_HUMAN,151,101,50,0.092100,0.449013,0.998928
4473,Q68DA7;FMN1_HUMAN,52,34,18,-0.616528,0.022413,0.998928
4472,Q96GC9;VMP1_HUMAN,29,18,11,-0.505624,0.296547,0.998928
4471,Q6UXH1;CREL2_HUMAN,81,55,26,0.192127,0.434800,0.998928
4470,P57729;RAB38_HUMAN,30,18,12,-0.147567,0.731418,0.998928
4469,Q9H2P0;ADNP_HUMAN,153,102,51,0.116910,0.231227,0.998928
4468,Q15582;BGH3_HUMAN,65,42,23,-0.084642,0.769554,0.998928
4467,P37840;SYUA_HUMAN,65,42,23,-0.913880,0.033232,0.998928


In [20]:

import numpy as np
import pandas as pd
from pathlib import Path

output_dir = Path("aligned_omics")
output_dir.mkdir(exist_ok=True)

# Examine the overall statistical results
print("Proteins tested:", len(results_df))
print("Unadjusted p < 0.05:",
      (results_df["p_value"] < 0.05).sum())
print("FDR < 0.05:",
      (results_df["FDR"] < 0.05).sum())
print("FDR < 0.10:",
      (results_df["FDR"] < 0.10).sum())

# Convert protein identifiers to candidate gene symbols
# Example: O00571;DDX3X_HUMAN -> DDX3X
ranked = results_df.copy()

ranked["Gene"] = (
    ranked["Protein"].astype(str)
    .str.split(";").str[-1]
    .str.replace(r"_HUMAN$", "", regex=True)
)

# Signed ranking score: positive = higher in female-derived lines
ranked["ranking_score"] = (
    np.sign(ranked["Female_minus_Male"])
    * -np.log10(ranked["p_value"].clip(lower=1e-300))
)

ranked = ranked.replace(
    [np.inf, -np.inf], np.nan
).dropna(subset=["ranking_score"])

# Keep ranking unambiguous at the gene-symbol level
ranked = ranked.sort_values(
    "ranking_score",
    key=lambda x: x.abs(),
    ascending=False
).drop_duplicates("Gene")

ranked = ranked.sort_values(
    "ranking_score", ascending=False
)

ranked[["Gene", "ranking_score"]].to_csv(
    output_dir / "sex_protein_exploratory_ranking.csv",
    index=False
)

print("\nRanked genes:", len(ranked))
print("\nTop-ranked candidates:")
display(
    ranked[["Gene", "ranking_score", "p_value", "FDR"]]
    .head(15)
)

print("\nRanking saved successfully!")


Proteins tested: 6705
Unadjusted p < 0.05: 216
FDR < 0.05: 0
FDR < 0.10: 0

Ranked genes: 6705

Top-ranked candidates:


,Gene,ranking_score,p_value,FDR
4133,DDX3X,4.048798,0.000089,0.599240
294,GNAS1,3.200051,0.000631,0.998928
3266,ROBO1,3.123386,0.000753,0.998928
6675,TECT3,2.667613,0.002150,0.998928
4377,MXRA7,2.645491,0.002262,0.998928
2985,HDHD1,2.635258,0.002316,0.998928
3721,EDIL3,2.570617,0.002688,0.998928
3878,DOCK2,2.557064,0.002773,0.998928
2896,PTPRU,2.490118,0.003235,0.998928
2480,KRT84,2.464211,0.003434,0.998928



Ranking saved successfully!


In [21]:

import numpy as np
import pandas as pd
from scipy.stats import t
from pathlib import Path

# Copy the existing statistical results
gsea_ranked = results_df.copy()

# Extract candidate gene symbols from protein identifiers
gsea_ranked["Gene"] = (
    gsea_ranked["Protein"].astype(str)
    .str.split(";").str[-1]
    .str.replace(r"_HUMAN$", "", regex=True)
)

# Number of parameters in the tissue-adjusted model
n_parameters = design.shape[1]

# Residual degrees of freedom
gsea_ranked["df_resid"] = (
    gsea_ranked["n"] - n_parameters
)

# Reconstruct the signed t-statistic from p-values
gsea_ranked["t_statistic"] = (
    np.sign(gsea_ranked["Female_minus_Male"])
    * t.isf(
        gsea_ranked["p_value"].clip(lower=1e-300) / 2,
        gsea_ranked["df_resid"]
    )
)

# Remove invalid rows
gsea_ranked = gsea_ranked.replace(
    [np.inf, -np.inf], np.nan
).dropna(subset=["Gene", "t_statistic"])

# Retain the strongest measurement per gene symbol
# when multiple proteins map to the same gene
gsea_ranked = (
    gsea_ranked
    .assign(abs_t=lambda df: df["t_statistic"].abs())
    .sort_values("abs_t", ascending=False)
    .drop_duplicates("Gene")
    .sort_values("t_statistic", ascending=False)
)

# Save ranked gene list
output_dir = Path("aligned_omics")
output_dir.mkdir(exist_ok=True)

gsea_ranked[["Gene", "t_statistic"]].to_csv(
    output_dir / "sex_proteomics_GSEA_ranking.csv",
    index=False
)

print("Genes ranked:", len(gsea_ranked))
print("Highest positive-ranked genes:")
display(
    gsea_ranked[
        ["Gene", "t_statistic", "FDR"]
    ].head(10)
)

print("\nRanking file saved!")


Genes ranked: 6705
Highest positive-ranked genes:


,Gene,t_statistic,FDR
4133,DDX3X,4.028672,0.599240
3266,ROBO1,3.572748,0.998928
294,GNAS1,3.492130,0.998928
6675,TECT3,3.270308,0.998928
4377,MXRA7,3.159045,0.998928
3721,EDIL3,3.141154,0.998928
2985,HDHD1,3.126574,0.998928
3878,DOCK2,3.082926,0.998928
5007,FA83D,3.056154,0.998928
2896,PTPRU,3.025561,0.998928



Ranking file saved!


In [22]:

import pandas as pd

xy_genes = [
    "XIST", "RPS4Y1", "DDX3Y",
    "KDM5D", "UTY", "DDX3X"
]

# Find markers represented in the ranked proteomics results
xy_results = gsea_ranked[
    gsea_ranked["Gene"].isin(xy_genes)
][
    ["Gene", "t_statistic",
     "Female_minus_Male", "p_value", "FDR"]
].copy()

print("X/Y candidate genes checked:", len(xy_genes))
print("Available in proteomics ranking:", len(xy_results))

display(xy_results.sort_values(
    "t_statistic", ascending=False
))

# Save the results
xy_results.to_csv(
    "aligned_omics/XY_proteomics_marker_results.csv",
    index=False
)


X/Y candidate genes checked: 6
Available in proteomics ranking: 1


,Gene,t_statistic,Female_minus_Male,p_value,FDR
4133,DDX3X,4.028672,0.426344,0.000089,0.59924


In [23]:

import pandas as pd

# Use the ranking created in Step 19
ranking = gsea_ranked.copy()

# Check gene identifiers
print("Total ranked entries:", len(ranking))
print("Unique gene symbols:", ranking["Gene"].nunique())

print("Missing gene symbols:",
      ranking["Gene"].isna().sum())

print("Duplicate gene symbols:",
      ranking["Gene"].duplicated().sum())

# Check identifiers that may need additional verification
suspicious = ranking[
    ~ranking["Gene"].astype(str).str.fullmatch(
        r"[A-Za-z0-9._-]+"
    )
]

print("Unusual gene identifiers:", len(suspicious))

print("\nExample gene symbols:")
print(ranking["Gene"].head(20).tolist())

print("\nExamples requiring review:")
display(suspicious[["Protein", "Gene"]].head(10))


Total ranked entries: 6705
Unique gene symbols: 6705
Missing gene symbols: 0
Duplicate gene symbols: 0
Unusual gene identifiers: 0

Example gene symbols:
['DDX3X', 'ROBO1', 'GNAS1', 'TECT3', 'MXRA7', 'EDIL3', 'HDHD1', 'DOCK2', 'FA83D', 'PTPRU', 'CDS1', 'KRT84', 'EMC6', 'APC5', 'XXLT1', 'PDZ11', 'AP1S2', 'RT18C', 'P4R3B', 'RBM45']

Examples requiring review:


,Protein,Gene


In [25]:

import zipfile
from pathlib import Path

# Locate the Reactome ZIP file
zip_file = Path("ReactomePathways.gmt.zip")

# Check that the file exists
print("File found:", zip_file.exists())

# Inspect the ZIP contents
with zipfile.ZipFile(zip_file, "r") as z:
    print("\nFiles inside ZIP:")
    for filename in z.namelist():
        print(filename)

    # Extract the files
    output_dir = Path("reactome_gene_sets")
    output_dir.mkdir(exist_ok=True)

    z.extractall(output_dir)

print("\nExtraction complete!")

# Find extracted GMT files
gmt_files = list(output_dir.rglob("*.gmt"))

print("\nGMT files found:", len(gmt_files))

for file in gmt_files:
    print(file)

# Preview the first pathway
if gmt_files:
    with open(gmt_files[0], "r") as f:
        first_line = f.readline().strip().split("\t")

    print("\nFirst pathway:", first_line[0])
    print("Description:", first_line[1])
    print("Number of genes:", len(first_line[2:]))
    print("First 10 gene identifiers:", first_line[2:12])


File found: True

Files inside ZIP:
ReactomePathways.gmt

Extraction complete!

GMT files found: 1
reactome_gene_sets/ReactomePathways.gmt

First pathway: 2-LTR circle formation
Description: R-HSA-164843
Number of genes: 13
First 10 gene identifiers: ['BANF1', 'HMGA1', 'LIG4', 'PSIP1', 'XRCC4', 'XRCC5', 'XRCC6', 'gag', 'gag-pol', 'rev']


In [26]:

from pathlib import Path

# Read Reactome pathway definitions
gmt_file = Path(
    "reactome_gene_sets/ReactomePathways.gmt"
)

pathways = {}

with open(gmt_file, "r") as f:
    for line in f:
        parts = line.strip().split("\t")

        if len(parts) < 3:
            continue

        pathway_name = parts[0]
        genes = set(parts[2:])

        pathways[pathway_name] = genes

# Genes available in the proteomics ranking
ranked_genes = set(gsea_ranked["Gene"])

# Count overlap between each pathway and ranked genes
pathway_coverage = []

for pathway, genes in pathways.items():
    overlap_count = len(genes & ranked_genes)

    pathway_coverage.append({
        "Pathway": pathway,
        "Total_genes": len(genes),
        "Matched_genes": overlap_count
    })

coverage_df = pd.DataFrame(pathway_coverage)

print("Total Reactome pathways:", len(pathways))

print(
    "Pathways with >=15 matched genes:",
    (coverage_df["Matched_genes"] >= 15).sum()
)

print(
    "Pathways with >=10 matched genes:",
    (coverage_df["Matched_genes"] >= 10).sum()
)

print("\nTop pathways by gene coverage:")
display(
    coverage_df.sort_values(
        "Matched_genes", ascending=False
    ).head(15)
)

# Identify hormone-related pathways
hormone_pathways = coverage_df[
    coverage_df["Pathway"].str.contains(
        "estrogen|androgen|hormone|steroid",
        case=False,
        regex=True
    )
]

print("\nHormone-related Reactome pathways:")
display(
    hormone_pathways.sort_values(
        "Matched_genes", ascending=False
    ).head(20)
)


Total Reactome pathways: 2874
Pathways with >=15 matched genes: 516
Pathways with >=10 matched genes: 753

Top pathways by gene coverage:


,Pathway,Total_genes,Matched_genes
806,Disease,2517,616
2330,Signal Transduction,2623,555
1236,Immune System,2504,538
1516,Metabolism of proteins,2113,492
1498,Metabolism,2197,448
1251,Infectious disease,1701,433
1867,Post-translational protein modification,1475,407
2766,Viral Infection Pathways,1274,374
1111,Gene expression (Transcription),1552,334
1500,Metabolism of RNA,793,289



Hormone-related Reactome pathways:


,Pathway,Total_genes,Matched_genes
1519,Metabolism of steroids,155,33
926,Estrogen-dependent gene expression,121,31
1812,Peptide hormone metabolism,86,15
949,Extra-nuclear estrogen signaling,75,15
1180,HSP90 chaperone cycle for steroid hormone rece...,57,10
1154,Growth hormone receptor signaling,24,5
927,Estrogen-dependent nuclear events downstream o...,24,4
74,Activated PKN1 stimulates transcription of AR ...,35,3
1518,Metabolism of steroid hormones,36,2
2008,RUNX1 regulates estrogen receptor mediated tra...,6,1


In [27]:

!pip -q install gseapy


In [28]:

import gseapy as gp
import pandas as pd
from pathlib import Path

# Input files
gmt_file = "reactome_gene_sets/ReactomePathways.gmt"

# Use your tissue-adjusted protein ranking
ranking_data = gsea_ranked[
    ["Gene", "t_statistic"]
].copy()

ranking_data = ranking_data.dropna()
ranking_data = ranking_data.drop_duplicates("Gene")
ranking_data = ranking_data.sort_values(
    "t_statistic", ascending=False
)

# Create output directory
gsea_output = Path("aligned_omics/reactome_gsea")
gsea_output.mkdir(parents=True, exist_ok=True)

# Run preranked GSEA
gsea_result = gp.prerank(
    rnk=ranking_data,
    gene_sets=gmt_file,
    min_size=15,
    max_size=500,
    permutation_num=1000,
    weight=1.0,
    seed=42,
    threads=4,
    outdir=str(gsea_output),
    verbose=False
)

# Extract results
enrichment_results = gsea_result.res2d.copy()

# Sort by adjusted significance
enrichment_results = enrichment_results.sort_values(
    "FDR q-val",
    ascending=True
)

print("Pathways tested:", len(enrichment_results))

print(
    "Pathways with FDR < 0.05:",
    (pd.to_numeric(
        enrichment_results["FDR q-val"]
    ) < 0.05).sum()
)

print("\nTop enriched pathways:")

display(
    enrichment_results[
        ["Term", "NES", "NOM p-val", "FDR q-val"]
    ].head(20)
)

# Export results
enrichment_results.to_csv(
    gsea_output / "reactome_gsea_results.csv",
    index=False
)

print("\nGSEA completed and results saved!")


Pathways tested: 513
Pathways with FDR < 0.05: 2

Top enriched pathways:


,Term,NES,NOM p-val,FDR q-val
1,Negative regulation of the PI3K/AKT network,-2.033815,0.002262,0.024564
0,"PI5P, PP2A and IER3 Regulate PI3K/AKT Signaling",-2.082777,0.000000,0.026132
2,Metabolism of amino acids and derivatives,1.978012,0.000000,0.053240
5,RNA Polymerase II Transcription Termination,1.755633,0.005607,0.261604
6,Processing of Capped Intronless Pre-mRNA,1.755633,0.005607,0.261604
3,Processing of Intronless Pre-mRNAs,1.801211,0.003690,0.307342
9,MHC class II antigen presentation,1.683198,0.005217,0.309900
7,Regulation of mitotic cell cycle,1.692639,0.012522,0.331703
8,APC/C-mediated degradation of cell cycle proteins,1.692639,0.012522,0.331703
4,Chromosome Maintenance,-1.760006,0.005013,0.339021



GSEA completed and results saved!


In [29]:

import pandas as pd
from pathlib import Path

output_dir = Path("aligned_omics/reactome_gsea")
output_dir.mkdir(parents=True, exist_ok=True)

# Ensure statistical columns are numeric
gsea_table = enrichment_results.copy()

for col in ["NES", "NOM p-val", "FDR q-val"]:
    gsea_table[col] = pd.to_numeric(
        gsea_table[col], errors="coerce"
    )

# 1. Significant pathways
significant = gsea_table[
    gsea_table["FDR q-val"] < 0.05
].copy()

print("Significant pathways:", len(significant))

display(significant[
    ["Term", "NES", "NOM p-val", "FDR q-val"]
])

# 2. Hormone-associated pathways
hormone_terms = (
    "estrogen|androgen|hormone|steroid"
)

hormone_results = gsea_table[
    gsea_table["Term"].str.contains(
        hormone_terms,
        case=False,
        regex=True,
        na=False
    )
].sort_values("FDR q-val")

print("\nHormone-related pathways tested:",
      len(hormone_results))

display(hormone_results[
    ["Term", "NES", "NOM p-val", "FDR q-val"]
])

# 3. Identify leading-edge genes
print("\nLeading genes in significant pathways:")

if "Lead_genes" in significant.columns:
    display(significant[
        ["Term", "NES", "FDR q-val", "Lead_genes"]
    ])
else:
    print("Available columns:", significant.columns.tolist())

# 4. Export results
significant.to_csv(
    output_dir / "significant_pathways.csv",
    index=False
)

hormone_results.to_csv(
    output_dir / "hormone_pathway_results.csv",
    index=False
)

print("\nPathway tables saved!")


Significant pathways: 2


,Term,NES,NOM p-val,FDR q-val
1,Negative regulation of the PI3K/AKT network,-2.033815,0.002262,0.024564
0,"PI5P, PP2A and IER3 Regulate PI3K/AKT Signaling",-2.082777,0.000000,0.026132



Hormone-related pathways tested: 4


,Term,NES,NOM p-val,FDR q-val
100,Peptide hormone metabolism,1.242870,0.198214,0.764645
68,Extra-nuclear estrogen signaling,1.303917,0.153043,0.845650
189,Metabolism of steroids,1.080537,0.334459,0.847230
494,Estrogen-dependent gene expression,-0.585867,0.990698,1.000000



Leading genes in significant pathways:


,Term,NES,FDR q-val,Lead_genes
1,Negative regulation of the PI3K/AKT network,-2.033815,0.024564,IRS1;ERBB2;GRB2;NTF3;EGFR;IRAK4;FGF2;RHOG;IRAK...
0,"PI5P, PP2A and IER3 Regulate PI3K/AKT Signaling",-2.082777,0.026132,IRS1;ERBB2;GRB2;NTF3;EGFR;IRAK4;FGF2;RHOG;IRAK1



Pathway tables saved!


In [30]:

import pandas as pd
from pathlib import Path

# Get the two significant pathways
sig = enrichment_results.copy()

sig["FDR q-val"] = pd.to_numeric(
    sig["FDR q-val"], errors="coerce"
)

sig = sig[sig["FDR q-val"] < 0.05].copy()

# Convert leading-edge gene strings into sets
leading_sets = {}

for _, row in sig.iterrows():
    genes = set(
        str(row["Lead_genes"]).split(";")
    )
    genes.discard("")
    leading_sets[row["Term"]] = genes

# Display complete leading-edge genes
for pathway, genes in leading_sets.items():
    print("\nPathway:", pathway)
    print("Leading-edge genes:", len(genes))
    print(", ".join(sorted(genes)))

# Compare the two pathways
if len(leading_sets) == 2:
    names = list(leading_sets.keys())

    genes_1 = leading_sets[names[0]]
    genes_2 = leading_sets[names[1]]

    shared = genes_1 & genes_2
    union = genes_1 | genes_2

    print("\nShared leading-edge genes:", len(shared))
    print("Combined unique leading-edge genes:", len(union))

    print("Jaccard similarity:",
          round(len(shared) / len(union), 3))

    print("\nShared genes:")
    print(", ".join(sorted(shared)))

    # Save the shared genes
    pd.DataFrame({
        "Gene": sorted(shared)
    }).to_csv(
        "aligned_omics/reactome_gsea/"
        "shared_PI3K_AKT_leading_genes.csv",
        index=False
    )



Pathway: Negative regulation of the PI3K/AKT network
Leading-edge genes: 10
AKT2, EGFR, ERBB2, FGF2, GRB2, IRAK1, IRAK4, IRS1, NTF3, RHOG

Pathway: PI5P, PP2A and IER3 Regulate PI3K/AKT Signaling
Leading-edge genes: 9
EGFR, ERBB2, FGF2, GRB2, IRAK1, IRAK4, IRS1, NTF3, RHOG

Shared leading-edge genes: 9
Combined unique leading-edge genes: 10
Jaccard similarity: 0.9

Shared genes:
EGFR, ERBB2, FGF2, GRB2, IRAK1, IRAK4, IRS1, NTF3, RHOG


In [31]:

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

# Use the broader 344-cell-line subset
X = proteomics_analysis.copy()
meta = analysis_metadata.loc[X.index].copy()

female = (
    meta["final_sex"] == "Female"
).astype(float)

# Control for tissue type
tissue = pd.get_dummies(
    meta["Tissue_type"],
    drop_first=True,
    dtype=float
)

design_broad = sm.add_constant(
    pd.concat(
        [female.rename("Female"), tissue],
        axis=1
    )
)

broad_results = []

for protein in X.columns:
    y = pd.to_numeric(
        X[protein], errors="coerce"
    )

    valid = y.notna()

    # Require sufficient observed measurements
    n_female = int((valid & (female == 1)).sum())
    n_male = int((valid & (female == 0)).sum())

    if n_female < 10 or n_male < 10:
        continue

    d = design_broad.loc[valid]
    response = y.loc[valid]

    if response.nunique() < 2:
        continue

    if np.linalg.matrix_rank(d.to_numpy()) < d.shape[1]:
        continue

    model = sm.OLS(response, d).fit()

    broad_results.append({
        "Protein": protein,
        "n": int(valid.sum()),
        "Female_minus_Male": model.params["Female"],
        "t_statistic": model.tvalues["Female"],
        "p_value": model.pvalues["Female"]
    })

broad_results = pd.DataFrame(broad_results)

if broad_results.empty:
    raise ValueError("No proteins passed the filters.")

# Adjust for multiple comparisons
broad_results["FDR"] = multipletests(
    broad_results["p_value"],
    method="fdr_bh"
)[1]

print("Broader analysis samples:", len(X))
print("Proteins tested:", len(broad_results))
print(
    "Proteins with FDR < 0.05:",
    (broad_results["FDR"] < 0.05).sum()
)

# Prepare ranking with original model t-statistics
broad_results["Gene"] = (
    broad_results["Protein"].astype(str)
    .str.split(";").str[-1]
    .str.replace(r"_HUMAN$", "", regex=True)
)

broad_ranked = (
    broad_results
    .sort_values(
        "t_statistic",
        key=lambda s: s.abs(),
        ascending=False
    )
    .drop_duplicates("Gene")
    .sort_values("t_statistic", ascending=False)
)

broad_results.to_csv(
    "aligned_omics/sex_proteomics_broad_results.csv",
    index=False
)

print("Ranked genes:", len(broad_ranked))
print("\nTop-ranked genes:")
display(
    broad_ranked[
        ["Gene", "t_statistic", "p_value", "FDR"]
    ].head(10)
)


Broader analysis samples: 344
Proteins tested: 6874
Proteins with FDR < 0.05: 0
Ranked genes: 6874

Top-ranked genes:


,Gene,t_statistic,p_value,FDR
4182,DDX3X,3.699918,0.000252,0.604326
219,SDHA,3.429938,0.000679,0.604326
2626,CERS2,3.356523,0.000881,0.604326
5075,FA83D,3.281628,0.001585,0.604326
3351,PTN1,3.266251,0.001203,0.604326
1922,PTN13,3.245533,0.001800,0.604326
721,FAM3C,3.186359,0.001580,0.604326
5610,CPNS1,3.139547,0.001843,0.604326
1098,SPTB2,3.131218,0.001895,0.604326
1020,RT26,3.123746,0.001943,0.604326


In [32]:

import gseapy as gp
import pandas as pd
from pathlib import Path

# Create output folder
broad_output = Path("aligned_omics/reactome_gsea_broad")
broad_output.mkdir(parents=True, exist_ok=True)

# Prepare ranked genes from the broader analysis
broad_ranking = broad_ranked[
    ["Gene", "t_statistic"]
].dropna().drop_duplicates("Gene")

broad_ranking = broad_ranking.sort_values(
    "t_statistic", ascending=False
)

# Run GSEA using the same parameters as before
broad_gsea = gp.prerank(
    rnk=broad_ranking,
    gene_sets="reactome_gene_sets/ReactomePathways.gmt",
    min_size=15,
    max_size=500,
    permutation_num=1000,
    weight=1.0,
    seed=42,
    threads=4,
    outdir=str(broad_output),
    verbose=False
)

# Save results
broad_enrichment = broad_gsea.res2d.copy()

for col in ["NES", "NOM p-val", "FDR q-val"]:
    broad_enrichment[col] = pd.to_numeric(
        broad_enrichment[col], errors="coerce"
    )

broad_enrichment = broad_enrichment.sort_values(
    "FDR q-val"
)

print("Pathways tested:", len(broad_enrichment))
print(
    "Significant pathways (FDR < 0.05):",
    (broad_enrichment["FDR q-val"] < 0.05).sum()
)

print("\nTop 15 pathways:")
display(
    broad_enrichment[
        ["Term", "NES", "NOM p-val", "FDR q-val"]
    ].head(15)
)

broad_enrichment.to_csv(
    broad_output / "reactome_gsea_broad_results.csv",
    index=False
)

# Compare PI3K/AKT pathways between analyses
target_pathways = [
    "Negative regulation of the PI3K/AKT network",
    "PI5P, PP2A and IER3 Regulate PI3K/AKT Signaling"
]

original = enrichment_results[
    enrichment_results["Term"].isin(target_pathways)
][["Term", "NES", "FDR q-val"]].copy()

broader = broad_enrichment[
    broad_enrichment["Term"].isin(target_pathways)
][["Term", "NES", "FDR q-val"]].copy()

comparison = original.merge(
    broader,
    on="Term",
    how="outer",
    suffixes=("_high_conf", "_broad")
)

print("\nPI3K/AKT pathway comparison:")
display(comparison)

comparison.to_csv(
    broad_output / "PI3K_AKT_sensitivity_comparison.csv",
    index=False
)

print("\nBroader GSEA completed!")


Pathways tested: 528
Significant pathways (FDR < 0.05): 29

Top 15 pathways:


,Term,NES,NOM p-val,FDR q-val
0,Cotranslational targeting of proteins to the e...,2.344179,0.000000,0.000000
1,Asparagine N-linked glycosylation,2.324364,0.000000,0.000000
2,Protein localization,2.224623,0.000000,0.001507
4,mRNA Splicing - Major Pathway,-2.164819,0.000000,0.005502
6,Transport to the Golgi and subsequent modifica...,2.054984,0.000000,0.007231
5,ER to Golgi Anterograde Transport,2.058696,0.000000,0.008361
3,mRNA Splicing,-2.170325,0.000000,0.008804
8,Aerobic respiration and respiratory electron t...,2.009211,0.000000,0.008910
9,Peptide hormone metabolism,1.999189,0.000000,0.008926
7,Muscle contraction,2.013415,0.000000,0.010245



PI3K/AKT pathway comparison:


,Term,NES_high_conf,FDR q-val_high_conf,NES_broad,FDR q-val_broad
0,Negative regulation of the PI3K/AKT network,-2.033815,0.024564,-1.570197,0.202950
1,"PI5P, PP2A and IER3 Regulate PI3K/AKT Signaling",-2.082777,0.026132,-1.457052,0.306394



Broader GSEA completed!


In [33]:

import pandas as pd
from pathlib import Path

# Prepare high-confidence results
high = enrichment_results[
    ["Term", "NES", "FDR q-val"]
].copy()

high = high.rename(columns={
    "NES": "NES_high",
    "FDR q-val": "FDR_high"
})

# Prepare broader results
broad = broad_enrichment[
    ["Term", "NES", "FDR q-val"]
].copy()

broad = broad.rename(columns={
    "NES": "NES_broad",
    "FDR q-val": "FDR_broad"
})

# Combine both analyses
comparison = high.merge(
    broad,
    on="Term",
    how="outer"
)

for col in [
    "NES_high", "FDR_high",
    "NES_broad", "FDR_broad"
]:
    comparison[col] = pd.to_numeric(
        comparison[col], errors="coerce"
    )

comparison["Significant_high"] = (
    comparison["FDR_high"] < 0.05
)

comparison["Significant_broad"] = (
    comparison["FDR_broad"] < 0.05
)

comparison["Significant_both"] = (
    comparison["Significant_high"] &
    comparison["Significant_broad"]
)

comparison["Same_direction"] = (
    comparison["NES_high"] *
    comparison["NES_broad"] > 0
)

print("Pathways compared:", len(comparison))

print(
    "Significant in both:",
    comparison["Significant_both"].sum()
)

print(
    "Same enrichment direction:",
    comparison["Same_direction"].sum()
)

print("\nPathways significant in either analysis:")

selected = comparison[
    comparison["Significant_high"] |
    comparison["Significant_broad"]
].sort_values("FDR_broad")

display(selected[
    [
        "Term",
        "NES_high",
        "FDR_high",
        "NES_broad",
        "FDR_broad",
        "Significant_both",
        "Same_direction"
    ]
])

# Export
output_dir = Path("aligned_omics/reactome_gsea")
output_dir.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    output_dir / "GSEA_sensitivity_comparison.csv",
    index=False
)

print("\nComparison saved successfully!")


Pathways compared: 536
Significant in both: 0
Same enrichment direction: 338

Pathways significant in either analysis:


,Term,NES_high,FDR_high,NES_broad,FDR_broad,Significant_both,Same_direction
30,Asparagine N-linked glycosylation,0.841734,1.000000,2.324364,0.000000,False,True
77,Cotranslational targeting of proteins to the e...,1.393729,0.712971,2.344179,0.000000,False,True
331,Protein localization,1.269571,0.813241,2.224623,0.001507,False,True
526,mRNA Splicing - Major Pathway,-0.901361,1.000000,-2.164819,0.005502,False,True
512,Transport to the Golgi and subsequent modifica...,0.824860,1.000000,2.054984,0.007231,False,True
123,ER to Golgi Anterograde Transport,0.978141,0.942016,2.058696,0.008361,False,True
525,mRNA Splicing,-0.925216,1.000000,-2.170325,0.008804,False,True
18,Aerobic respiration and respiratory electron t...,1.594175,0.529822,2.009211,0.008910,False,True
309,Peptide hormone metabolism,1.242870,0.764645,1.999189,0.008926,False,True
270,Muscle contraction,1.435776,0.802528,2.013415,0.010245,False,True



Comparison saved successfully!


In [34]:

import pandas as pd

# Use the five tissues from the high-confidence analysis
same_tissues = high_conf_tissues.tolist()

# Select all usable labels within those tissues
same_tissue_metadata = analysis_metadata[
    analysis_metadata["Tissue_type"].isin(same_tissues)
].copy()

same_tissue_ids = same_tissue_metadata.index.tolist()

same_tissue_proteomics = proteomics_analysis.loc[
    same_tissue_ids
]

print("Tissues:", len(same_tissues))
print("High-confidence samples:", len(high_conf_metadata))
print("All usable samples in same tissues:",
      len(same_tissue_metadata))

print("\nSex distribution:")
print(same_tissue_metadata["final_sex"].value_counts())

print("\nLabel confidence:")
print(same_tissue_metadata["label_confidence"].value_counts())

print("\nSamples per tissue and sex:")
display(pd.crosstab(
    same_tissue_metadata["Tissue_type"],
    same_tissue_metadata["final_sex"]
))


Tissues: 5
High-confidence samples: 155
All usable samples in same tissues: 278

Sex distribution:
final_sex
Male      175
Female    103
Name: count, dtype: int64

Label confidence:
label_confidence
high    155
low     123
Name: count, dtype: int64

Samples per tissue and sex:


final_sex,Female,Male
Tissue_type,,
Central Nervous System,11,15
Haematopoietic and Lymphoid,33,45
Large Intestine,13,19
Lung,34,83
Skin,12,13


In [35]:

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from pathlib import Path

# Use the same five tissues with all usable labels
X = same_tissue_proteomics.copy()
meta = same_tissue_metadata.loc[X.index]

female = (meta["final_sex"] == "Female").astype(float)

# Adjust for tissue differences
tissue = pd.get_dummies(
    meta["Tissue_type"],
    drop_first=True,
    dtype=float
)

design_same = sm.add_constant(
    pd.concat(
        [female.rename("Female"), tissue],
        axis=1
    )
)

same_results = []

for protein in X.columns:
    y = pd.to_numeric(X[protein], errors="coerce")
    valid = y.notna()

    n_female = int((valid & (female == 1)).sum())
    n_male = int((valid & (female == 0)).sum())

    if n_female < 10 or n_male < 10:
        continue

    d = design_same.loc[valid]
    response = y.loc[valid]

    if response.nunique() < 2:
        continue

    if np.linalg.matrix_rank(d.to_numpy()) < d.shape[1]:
        continue

    model = sm.OLS(response, d).fit()

    same_results.append({
        "Protein": protein,
        "n": int(valid.sum()),
        "Female_minus_Male": model.params["Female"],
        "t_statistic": model.tvalues["Female"],
        "p_value": model.pvalues["Female"]
    })

same_results = pd.DataFrame(same_results)

if same_results.empty:
    raise ValueError("No proteins passed filtering.")

same_results["FDR"] = multipletests(
    same_results["p_value"],
    method="fdr_bh"
)[1]

# Convert protein identifiers to candidate gene names
same_results["Gene"] = (
    same_results["Protein"].astype(str)
    .str.split(";").str[-1]
    .str.replace(r"_HUMAN$", "", regex=True)
)

# Prepare ranked genes
same_ranked = (
    same_results
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["t_statistic"])
    .sort_values(
        "t_statistic",
        key=lambda s: s.abs(),
        ascending=False
    )
    .drop_duplicates("Gene")
    .sort_values("t_statistic", ascending=False)
)

output_dir = Path("aligned_omics")
output_dir.mkdir(exist_ok=True)

same_results.to_csv(
    output_dir / "sex_proteomics_same_tissues.csv",
    index=False
)

print("Samples:", len(X))
print("Proteins tested:", len(same_results))
print("Proteins FDR < 0.05:",
      (same_results["FDR"] < 0.05).sum())
print("Ranked genes:", len(same_ranked))

display(same_ranked[
    ["Gene", "t_statistic", "p_value", "FDR"]
].head(10))


Samples: 278
Proteins tested: 7206
Proteins FDR < 0.05: 0
Ranked genes: 7206


,Gene,t_statistic,p_value,FDR
4322,DDX3X,3.733704,0.000230,0.872427
2117,AT131,3.276914,0.001188,0.872427
1361,IVD,3.232233,0.001391,0.872427
223,SDHA,3.224473,0.001416,0.872427
1006,ESYT1,3.165572,0.001724,0.872427
3462,PTN1,3.061523,0.002423,0.872427
272,SYTM,2.946460,0.003493,0.872427
1336,RM39,2.927756,0.003703,0.872427
1039,RT26,2.924954,0.003738,0.872427
4514,CDS1,2.881141,0.004642,0.872427


In [36]:

import gseapy as gp
import pandas as pd
from pathlib import Path

same_output = Path("aligned_omics/reactome_gsea_same_tissues")
same_output.mkdir(parents=True, exist_ok=True)

same_gsea = gp.prerank(
    rnk=same_ranked[["Gene", "t_statistic"]],
    gene_sets="reactome_gene_sets/ReactomePathways.gmt",
    min_size=15,
    max_size=500,
    permutation_num=1000,
    weight=1.0,
    seed=42,
    threads=4,
    outdir=str(same_output),
    verbose=False
)

same_enrichment = same_gsea.res2d.copy()

for col in ["NES", "FDR q-val"]:
    same_enrichment[col] = pd.to_numeric(
        same_enrichment[col], errors="coerce"
    )

print("Pathways tested:", len(same_enrichment))
print(
    "Significant pathways:",
    (same_enrichment["FDR q-val"] < 0.05).sum()
)

# Compare pathways particularly relevant to our findings
targets = [
    "Negative regulation of the PI3K/AKT network",
    "PI5P, PP2A and IER3 Regulate PI3K/AKT Signaling",
    "Peptide hormone metabolism"
]

comparison_tables = []

for name, data in [
    ("155_high", enrichment_results),
    ("278_same_tissues", same_enrichment),
    ("344_broad", broad_enrichment)
]:
    subset = data[data["Term"].isin(targets)].copy()
    subset = subset[["Term", "NES", "FDR q-val"]]
    subset["Analysis"] = name
    comparison_tables.append(subset)

comparison = pd.concat(
    comparison_tables,
    ignore_index=True
)

comparison["NES"] = pd.to_numeric(comparison["NES"])
comparison["FDR q-val"] = pd.to_numeric(
    comparison["FDR q-val"]
)

display(comparison.sort_values(["Term", "Analysis"]))

same_enrichment.to_csv(
    same_output / "gsea_results.csv",
    index=False
)

comparison.to_csv(
    same_output / "three_group_comparison.csv",
    index=False
)

print("GSEA comparison saved!")


Pathways tested: 548
Significant pathways: 6


,Term,NES,FDR q-val,Analysis
0,Negative regulation of the PI3K/AKT network,-2.033815,0.024564,155_high
3,Negative regulation of the PI3K/AKT network,-1.736632,0.267081,278_same_tissues
7,Negative regulation of the PI3K/AKT network,-1.570197,0.202950,344_broad
1,"PI5P, PP2A and IER3 Regulate PI3K/AKT Signaling",-2.082777,0.026132,155_high
4,"PI5P, PP2A and IER3 Regulate PI3K/AKT Signaling",-1.668609,0.362412,278_same_tissues
8,"PI5P, PP2A and IER3 Regulate PI3K/AKT Signaling",-1.457052,0.306394,344_broad
2,Peptide hormone metabolism,1.242870,0.764645,155_high
5,Peptide hormone metabolism,1.482486,0.273972,278_same_tissues
6,Peptide hormone metabolism,1.999189,0.008926,344_broad


GSEA comparison saved!


In [37]:

# Identify significant pathways in the 278-sample analysis
significant_same = same_enrichment[
    same_enrichment["FDR q-val"] < 0.05
].copy()

significant_same = significant_same.sort_values("FDR q-val")

print("Significant pathways:", len(significant_same))

display(
    significant_same[
        ["Term", "NES", "NOM p-val", "FDR q-val"]
    ]
)

# Compare these pathways across all three analyses
important_terms = significant_same["Term"].tolist()

tables = []

for name, df in [
    ("155_high", enrichment_results),
    ("278_same_tissues", same_enrichment),
    ("344_broad", broad_enrichment)
]:
    subset = df[df["Term"].isin(important_terms)].copy()
    subset["Analysis"] = name
    tables.append(subset)

final_check = pd.concat(tables, ignore_index=True)

display(
    final_check[
        ["Term", "Analysis", "NES", "FDR q-val"]
    ].sort_values(["Term", "Analysis"])
)


Significant pathways: 6


,Term,NES,NOM p-val,FDR q-val
0,Cotranslational targeting of proteins to the e...,2.454717,0.000000,0.000000
1,Protein localization,2.209564,0.000000,0.002905
2,Asparagine N-linked glycosylation,2.100342,0.000000,0.008068
3,Aerobic respiration and respiratory electron t...,2.064422,0.000000,0.011134
4,COPII-mediated vesicle transport,2.008579,0.002114,0.017621
5,Respiratory electron transport,1.924548,0.001953,0.037759


,Term,Analysis,NES,FDR q-val
0,Aerobic respiration and respiratory electron t...,155_high,1.594175,0.529822
9,Aerobic respiration and respiratory electron t...,278_same_tissues,2.064422,0.011134
15,Aerobic respiration and respiratory electron t...,344_broad,2.009211,0.008910
4,Asparagine N-linked glycosylation,155_high,0.841734,1.000000
8,Asparagine N-linked glycosylation,278_same_tissues,2.100342,0.008068
13,Asparagine N-linked glycosylation,344_broad,2.324364,0.000000
5,COPII-mediated vesicle transport,155_high,-0.622574,1.000000
10,COPII-mediated vesicle transport,278_same_tissues,2.008579,0.017621
16,COPII-mediated vesicle transport,344_broad,1.889754,0.019887
1,Cotranslational targeting of proteins to the e...,155_high,1.393729,0.712971


In [38]:

import pandas as pd
from pathlib import Path

folder = Path("aligned_omics")

files = {
    "Expression": "expression_aligned.csv",
    "Mutation": "mutation_aligned.csv",
    "Copy number": "cnv_aligned.csv",
    "Proteomics": "proteomics_aligned.csv",
    "Sample metadata": "sample_metadata.csv",
    "Sex labels": "sex_labels_matched.csv",
    "X/Y module": "XY_gene_module_candidates.csv"
}

for name, filename in files.items():
    path = folder / filename

    if path.exists():
        df = pd.read_csv(path)
        print(f"{name}: FOUND | Shape: {df.shape}")
    else:
        print(f"{name}: MISSING")


Expression: FOUND | Shape: (526, 18026)
Mutation: FOUND | Shape: (526, 1668)
Copy number: FOUND | Shape: (526, 23265)
Proteomics: FOUND | Shape: (526, 8499)
Sample metadata: FOUND | Shape: (526, 7)
Sex labels: FOUND | Shape: (526, 30)
X/Y module: FOUND | Shape: (5, 1)


In [39]:

import pandas as pd
from pathlib import Path

folder = Path("aligned_omics")

# Load the four aligned datasets
matrix_files = {
    "expression": "expression_aligned.csv",
    "mutation": "mutation_aligned.csv",
    "cnv": "cnv_aligned.csv",
    "proteomics": "proteomics_aligned.csv"
}

matrices_check = {}

for name, filename in matrix_files.items():
    df = pd.read_csv(folder / filename, index_col=0)
    df.index = df.index.astype(str)
    matrices_check[name] = df

# Use expression as the reference
reference_ids = matrices_check["expression"].index

print("Reference samples:", len(reference_ids))
print("Unique reference IDs:", reference_ids.is_unique)

for name, df in matrices_check.items():
    print(
        f"{name}:",
        "Shape:", df.shape,
        "| IDs unique:", df.index.is_unique,
        "| Same sample order:", df.index.equals(reference_ids)
    )

# Confirm the X/Y candidate list
xy_module = pd.read_csv(
    folder / "XY_gene_module_candidates.csv"
)

print("\nX/Y candidate genes:")
print(xy_module.iloc[:, 0].tolist())

# Verify all five genes are available in expression
xy_genes = xy_module.iloc[:, 0].astype(str).tolist()

print(
    "\nAll X/Y genes in expression:",
    all(gene in matrices_check["expression"].columns
        for gene in xy_genes)
)

# Confirm the sex-label table covers the same samples
sex_check = pd.read_csv(
    folder / "sex_labels_matched.csv"
)

print("\nSex metadata rows:", len(sex_check))
print("Usable sex labels:", sex_check["usable_sex"].sum())


Reference samples: 526
Unique reference IDs: True
expression: Shape: (526, 18025) | IDs unique: True | Same sample order: True
mutation: Shape: (526, 1667) | IDs unique: True | Same sample order: True
cnv: Shape: (526, 23264) | IDs unique: True | Same sample order: True
proteomics: Shape: (526, 8498) | IDs unique: True | Same sample order: True

X/Y candidate genes:
['XIST', 'RPS4Y1', 'DDX3Y', 'KDM5D', 'UTY']

All X/Y genes in expression: True

Sex metadata rows: 526
Usable sex labels: 492


In [40]:

from pathlib import Path
import zipfile

folder = Path("aligned_omics")
package = Path("week2_aligned_omics_package.zip")

files_to_include = [
    "expression_aligned.csv",
    "mutation_aligned.csv",
    "cnv_aligned.csv",
    "proteomics_aligned.csv",
    "sample_metadata.csv",
    "sex_labels_matched.csv",
    "XY_gene_module_candidates.csv"
]

readme = """Week 2 Multi-Omics Dataset Package

CONTENTS
- expression_aligned.csv: 526 cell lines x 18,025 features
- mutation_aligned.csv: 526 cell lines x 1,667 features
- cnv_aligned.csv: 526 cell lines x 23,264 features
- proteomics_aligned.csv: 526 cell lines x 8,498 features
- sample_metadata.csv: cell-line metadata
- sex_labels_matched.csv: sex-label matching metadata
- XY_gene_module_candidates.csv: five X/Y candidate genes

IDENTIFIERS
- Molecular matrices use SIDM identifiers as row indices.
- All four molecular matrices have identical sample ordering.
- Sex labels are provisional name-based matches.
- 492 of 526 cell lines have usable sex labels.

IMPORTANT
- These matrices are ALIGNED, not fully preprocessed.
- Proteomics contains missing measurements.
- Fit imputation, feature selection, and scaling on
  training data only.
- Apply fitted transformations to validation/test data.
- Do not use this package as evidence of verified
  sex-associated biological differences.

X/Y CANDIDATES
XIST, RPS4Y1, DDX3Y, KDM5D, UTY
"""

# Verify required files before packaging
missing = [
    name for name in files_to_include
    if not (folder / name).exists()
]

if missing:
    raise FileNotFoundError(
        f"Missing files: {missing}"
    )

# Create ZIP without changing source files
with zipfile.ZipFile(
    package, "w", zipfile.ZIP_DEFLATED
) as z:
    for name in files_to_include:
        z.write(folder / name, arcname=name)

    z.writestr("README.txt", readme)

print("Package created:", package)
print("Size (MB):", round(
    package.stat().st_size / (1024**2), 2
))

print("\nFiles included:")
with zipfile.ZipFile(package) as z:
    for name in z.namelist():
        print("-", name)


Package created: week2_aligned_omics_package.zip
Size (MB): 26.62

Files included:
- expression_aligned.csv
- mutation_aligned.csv
- cnv_aligned.csv
- proteomics_aligned.csv
- sample_metadata.csv
- sex_labels_matched.csv
- XY_gene_module_candidates.csv
- README.txt


In [41]:

%%writefile multiomics_preprocessing_pipeline.py

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


class MultiOmicsPreprocessor:
    def __init__(self, protein_min_coverage=0.70):
        self.protein_min_coverage = protein_min_coverage
        self.models = {}

    def fit(self, matrices, train_ids):
        """Fit preprocessing on training samples only."""
        self.models = {}

        for name, df in matrices.items():
            train = df.loc[train_ids].apply(
                pd.to_numeric, errors="coerce"
            )

            # Proteomics: remove low-coverage proteins
            if name == "proteomics":
                coverage = train.notna().mean()
                selected = coverage[
                    coverage >= self.protein_min_coverage
                ].index
            else:
                selected = train.columns[
                    train.notna().any(axis=0)
                ]

            train = train[selected]

            # Learn missing-value replacements
            imputer = SimpleImputer(
                strategy="median",
                keep_empty_features=True
            )
            filled = imputer.fit_transform(train)

            # Remove constant features using training data
            variance = filled.var(axis=0)
            keep = variance > 0

            selected = selected[keep]
            filled = filled[:, keep]

            if len(selected) == 0:
                raise ValueError(
                    f"No features retained for {name}"
                )

            # Scale continuous layers only
            scaler = None

            if name in ["expression", "proteomics"]:
                scaler = StandardScaler()
                scaler.fit(filled)

            self.models[name] = {
                "features": selected.tolist(),
                "imputer": imputer,
                "keep": keep,
                "scaler": scaler
            }

        return self

    def transform(self, matrices, sample_ids):
        """Apply training-fitted transformations."""
        output = {}

        for name, model in self.models.items():
            df = matrices[name].loc[sample_ids]

            # Use columns retained during fitting
            selected_features = model["features"]

            # Imputer was fitted before constant filtering
            original_features = (
                model["imputer"].feature_names_in_
            )

            data = df.reindex(
                columns=original_features
            ).apply(pd.to_numeric, errors="coerce")

            values = model["imputer"].transform(data)
            values = values[:, model["keep"]]

            if model["scaler"] is not None:
                values = model["scaler"].transform(values)

            output[name] = pd.DataFrame(
                values,
                index=sample_ids,
                columns=selected_features
            )

        return output

    def fit_transform(self, matrices, train_ids):
        self.fit(matrices, train_ids)
        return self.transform(matrices, train_ids)


Writing multiomics_preprocessing_pipeline.py


In [42]:

from multiomics_preprocessing_pipeline import MultiOmicsPreprocessor

# Reuse the previously verified aligned matrices
matrices = matrices_check

# Small temporary subset for a functional test
test_train_ids = list(
    matrices["expression"].index[:30]
)

test_eval_ids = list(
    matrices["expression"].index[30:40]
)

# Fit on temporary training samples
processor = MultiOmicsPreprocessor(
    protein_min_coverage=0.70
)

processor.fit(matrices, test_train_ids)

# Apply the same transformations to both groups
train_test = processor.transform(
    matrices, test_train_ids
)

eval_test = processor.transform(
    matrices, test_eval_ids
)

# Verify outputs
for name in matrices:
    train_df = train_test[name]
    eval_df = eval_test[name]

    print(f"\n{name}")
    print("Train shape:", train_df.shape)
    print("Evaluation shape:", eval_df.shape)
    print(
        "Same features:",
        train_df.columns.equals(eval_df.columns)
    )
    print(
        "Missing values:",
        int(train_df.isna().sum().sum()),
        int(eval_df.isna().sum().sum())
    )



expression
Train shape: (30, 13603)
Evaluation shape: (10, 13603)
Same features: True
Missing values: 0 0

mutation
Train shape: (30, 964)
Evaluation shape: (10, 964)
Same features: True
Missing values: 0 0

cnv
Train shape: (30, 18755)
Evaluation shape: (10, 18755)
Same features: True
Missing values: 0 0

proteomics
Train shape: (30, 4548)
Evaluation shape: (10, 4548)
Same features: True
Missing values: 0 0


In [43]:

import numpy as np
import pandas as pd

for name in matrices:
    train = train_test[name]
    evaluation = eval_test[name]

    print(f"\n--- {name.upper()} ---")

    # Check for infinite or missing values
    print(
        "All training values finite:",
        np.isfinite(train.to_numpy()).all()
    )

    print(
        "All evaluation values finite:",
        np.isfinite(evaluation.to_numpy()).all()
    )

    # Confirm feature dimensions match
    print(
        "Feature alignment:",
        train.columns.equals(evaluation.columns)
    )

    # Check scaling of continuous layers
    if name in ["expression", "proteomics"]:
        max_mean = train.mean().abs().max()

        print(
            "Largest absolute training-feature mean:",
            round(float(max_mean), 8)
        )

    # Confirm original data is still aligned
    print(
        "Original sample count:",
        len(matrices[name])
    )



--- EXPRESSION ---
All training values finite: True
All evaluation values finite: True
Feature alignment: True
Largest absolute training-feature mean: 0.0
Original sample count: 526

--- MUTATION ---
All training values finite: True
All evaluation values finite: True
Feature alignment: True
Original sample count: 526

--- CNV ---
All training values finite: True
All evaluation values finite: True
Feature alignment: True
Original sample count: 526

--- PROTEOMICS ---
All training values finite: True
All evaluation values finite: True
Feature alignment: True
Largest absolute training-feature mean: 0.0
Original sample count: 526
